# VLM scene extraction: pilot, then full shards
Attach MSCOCO images and dataset_coco.json. Enable GPU and Internet. Start with SMOKE=True (100 images). Inspect the scene file and measure seconds/image before full extraction. VLM receives images only; references are never sent to it.


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path('/kaggle/working/Image_Captioning')
def run(args):
    print('Running:', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True)
if not REPO.exists():
    run(['git', 'clone', 'https://github.com/Supzxjee/Image_Captioning.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', 'main'])
run(['git', 'pull', '--ff-only'])
run(['git', 'rev-parse', 'HEAD'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-vlm.txt'])


In [ ]:
JSON = '/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json'
IMAGES = '/kaggle/input/datasets/vuthetam/mscoco-2014/images'
assert Path(JSON).is_file(), JSON
assert Path(IMAGES).is_dir(), IMAGES
SMOKE = True
PART = 1
PARTS = 8  # Adjust only BEFORE full extraction, based on pilot speed.
REVISION = 'main'  # For full shards, copy resolved_revision from the pilot .meta.json.
OUT = Path('/kaggle/working/vlm_outputs')
OUT.mkdir(exist_ok=True)
SCENES = OUT / (f'scenes_part_{PART:02d}_of_{PARTS:02d}' + ('_smoke' if SMOKE else '') + '.jsonl')
run([sys.executable, '-u', 'build_vlm_prompts.py', 'generate',
     '--dataset-json-path', JSON, '--base-path', IMAGES,
     '--part', PART, '--parts', PARTS, '--limit', 100 if SMOKE else 0,
     '--revision', REVISION, '--output', SCENES])


In [ ]:
import json
rows = [json.loads(line) for line in SCENES.read_text().splitlines() if line.strip()]
for row in rows[:10]:
    print(row['filename'], row['scene'])
print('Saved scenes:', len(rows))
print(SCENES.with_suffix('.jsonl.meta.json').read_text())
# Full shards: save every .jsonl and .meta.json, then attach all shards to the embedding notebook.


In [ ]:
# Pilot embedding checks both cache formats; these partial caches cannot train the full split.
if SMOKE:
    run([sys.executable, '-u', 'build_vlm_prompts.py', 'embed',
         '--dataset-json-path', JSON, '--scenes', SCENES,
         '--output-dir', OUT / 'smoke_embeddings', '--allow-partial'])
